# 05 — Experimento 4: Processamento de Linguagem Natural (PLN)

Atende às **Seções 4.4, 5.2 e 6.2 (Experimento 4)**:
- Limpeza textual, normalização de acentos, preservação de negações agronômicas (*'sem manchas'*) e extração de sintomas.
- Comparação com Validação Cruzada Estratificada (5-Fold CV) entre:
  1. **TF-IDF + LogisticRegression**
  2. **TF-IDF + LinearSVC Calibrado**

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.nlp.symptom_classifier import SymptomClassifier

exp4 = json.loads((ROOT / 'results' / 'metrics' / 'exp4_nlp_models.json').read_text(encoding='utf-8'))
df_comp = pd.DataFrame([
    exp4['logistic_regression_5fold_cv'],
    exp4['calibrated_linear_svm_5fold_cv']
])[['algorithm', 'num_samples', 'accuracy', 'precision_macro', 'recall_macro', 'f1_macro']]
display(df_comp)

,algorithm,num_samples,accuracy,precision_macro,recall_macro,f1_macro
0,TF-IDF + logistic_regression,60,0.95,0.9515,0.95,0.95
1,TF-IDF + svm,60,0.95,0.9515,0.95,0.95


In [2]:
# Demonstração de inferência e extração de sintomas sem inventar informações
clf = SymptomClassifier(algorithm='logistic_regression')
clf.train_from_csv(evaluate_cv=False)
exemplos = [
    'Folhas inferiores do tomateiro com anéis concêntricos tipo alvo e halo amarelado.',
    'Folha de batata com grandes manchas encharcadas escuras e mofo branco no verso.',
    'Folha de tomate totalmente verde, íntegra e sem manchas escuras.'
]
for txt in exemplos:
    out = clf.predict(txt)
    print(f"Texto: {txt}")
    print(f" -> Classe: {out['predicted_class']} (conf={out['confidence']:.2%}) | Sintomas: {out['metadata']['extracted_symptoms']}\n")

Texto: Folhas inferiores do tomateiro com anéis concêntricos tipo alvo e halo amarelado.
 -> Classe: Tomato___Early_blight (conf=90.15%) | Sintomas: ['aneis_concentricos', 'halo_amarelado']

Texto: Folha de batata com grandes manchas encharcadas escuras e mofo branco no verso.
 -> Classe: Potato___Late_blight (conf=91.07%) | Sintomas: ['lesoes_encharcadas', 'mofo_esbranquicado']

Texto: Folha de tomate totalmente verde, íntegra e sem manchas escuras.
 -> Classe: Tomato___Healthy (conf=86.42%) | Sintomas: ['folha_saudavel']

